# 03 - PySpark ALS Utility and Anonymization Feedback

Plug-and-play recommender baseline: use PySpark MLlib `ALS` instead of a custom NumPy implementation. Spark exposes the learned user/item factor tables, so we can still use factor norms as blue-team diagnostics.

Core references: Spark MLlib, Spark collaborative filtering docs, ALS/matrix factorization, and Netflix de-anonymization are cited in `notebooks/README.md`.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib-cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    from pyspark.ml.evaluation import RegressionEvaluator
    from pyspark.ml.recommendation import ALS
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
except ImportError as exc:
    raise ImportError("Install notebook ML deps with: python -m pip install -e '.[ml]'") from exc

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from guardrails_sensitive_data.anonymization import ReleaseData, add_time_and_rating_features, evaluate_release_linkage_risk, summarize_linkage_trials
from guardrails_sensitive_data.netflix_io import netflix_paths, read_movie_titles, read_netflix_ratings

DATA_DIR = ROOT / "data" / "netflix"
SEED = 333
MAX_ROWS = 250_000
if sns is not None:
    sns.set_theme(style="whitegrid")

In [ ]:
spark = (
    SparkSession.builder
    .appName("netflix-privacy-als")
    .master("local[*]")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
ratings = add_time_and_rating_features(read_netflix_ratings(DATA_DIR, max_rows=MAX_ROWS, include_date=True))
train_pd = ratings.sample(frac=0.8, random_state=SEED)
test_pd = ratings.drop(train_pd.index)
train_sdf = spark.createDataFrame(train_pd[["customer_id", "movie_id", "rating"]].astype({"customer_id": "int32", "movie_id": "int32", "rating": "float32"}))
test_sdf = spark.createDataFrame(test_pd[["customer_id", "movie_id", "rating"]].astype({"customer_id": "int32", "movie_id": "int32", "rating": "float32"}))
{"train_rows": train_sdf.count(), "test_rows": test_sdf.count()}

## Fit Spark ALS

`coldStartStrategy="drop"` avoids NaN predictions when a test user/item was absent from training.

In [ ]:
als = ALS(
    userCol="customer_id",
    itemCol="movie_id",
    ratingCol="rating",
    rank=32,
    maxIter=10,
    regParam=0.08,
    seed=SEED,
    coldStartStrategy="drop",
    nonnegative=False,
)
model = als.fit(train_sdf)
predictions = model.transform(test_sdf)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
{"rmse": evaluator.evaluate(predictions), "predicted_rows": predictions.count()}

## Interpret item factors for blue-team controls

Spark stores item factors in `model.itemFactors` with columns `id` and `features`. High factor norm, especially after adjusting for popularity, marks movies whose exact ratings may carry unusually specific taste information.

In [ ]:
def spark_item_factor_diagnostics(model, train_pd: pd.DataFrame) -> pd.DataFrame:
    factors = model.itemFactors.toPandas().rename(columns={"id": "movie_id"})
    factors["factor_norm"] = factors["features"].apply(lambda xs: float(np.linalg.norm(xs)))
    counts = train_pd["movie_id"].value_counts().rename("rating_count")
    means = train_pd.groupby("movie_id")["rating"].mean().rename("rating_mean")
    diag = factors.merge(counts, left_on="movie_id", right_index=True).merge(means, left_on="movie_id", right_index=True)
    diag["latent_leverage"] = diag["factor_norm"] / np.sqrt(diag["rating_count"].clip(lower=1))
    diag["popularity_percentile"] = diag["rating_count"].rank(pct=True)
    diag["leverage_percentile"] = diag["latent_leverage"].rank(pct=True)
    diag["privacy_weight"] = 0.65 * diag["leverage_percentile"] + 0.35 * (1 - diag["popularity_percentile"])
    return diag.sort_values("privacy_weight", ascending=False).reset_index(drop=True)

paths = netflix_paths(DATA_DIR)
titles = read_movie_titles(paths.movie_titles_file)
diagnostics = spark_item_factor_diagnostics(model, train_pd).merge(titles, on="movie_id", how="left")
diagnostics.head(15)[["movie_id", "title", "rating_count", "rating_mean", "factor_norm", "latent_leverage", "privacy_weight"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
if sns is not None:
    sns.scatterplot(data=diagnostics, x="rating_count", y="latent_leverage", hue="privacy_weight", palette="viridis", ax=ax, s=35)
else:
    scatter = ax.scatter(diagnostics["rating_count"], diagnostics["latent_leverage"], c=diagnostics["privacy_weight"], cmap="viridis", s=35)
    fig.colorbar(scatter, ax=ax, label="privacy_weight")
ax.set_xscale("log")
ax.set_title("Spark ALS latent leverage vs. popularity")
plt.tight_layout()

## ML-informed anonymization prototype

Use item-factor diagnostics as a non-isotropic noise schedule: high-risk movies get more rating noise and month generalization.

In [ ]:
def make_latent_sensitive_release(frame: pd.DataFrame, diagnostics: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    data = add_time_and_rating_features(frame).merge(diagnostics[["movie_id", "privacy_weight"]], on="movie_id", how="left")
    data["privacy_weight"] = data["privacy_weight"].fillna(0.5)
    flip_probability = 0.08 + 0.47 * data["privacy_weight"].clip(0, 1).to_numpy()
    flip = rng.random(len(data)) < flip_probability
    direction = rng.choice([-1, 1], size=len(data))
    noisy = data["rating"].to_numpy(dtype=np.int16).copy()
    noisy[flip] = np.clip(noisy[flip] + direction[flip], 1, 5)
    data["rating_ml_noisy"] = noisy.astype("int8")

    threshold = diagnostics["privacy_weight"].quantile(0.85)
    high_risk = data["privacy_weight"] >= threshold
    data["month_ml_generalized"] = data["month"].astype("string")
    data.loc[high_risk, "month_ml_generalized"] = data.loc[high_risk, "year"].astype("string")
    return data

ml_release = make_latent_sensitive_release(train_pd, diagnostics)
ml_release[["movie_id", "customer_id", "rating", "rating_ml_noisy", "month", "month_ml_generalized", "privacy_weight"]].head()

In [ ]:
ml_train_sdf = spark.createDataFrame(
    ml_release[["customer_id", "movie_id", "rating_ml_noisy"]]
    .rename(columns={"rating_ml_noisy": "rating"})
    .astype({"customer_id": "int32", "movie_id": "int32", "rating": "float32"})
)
noisy_model = als.fit(ml_train_sdf)
noisy_predictions = noisy_model.transform(test_sdf)
pd.DataFrame([
    {"training_release": "original_train", "rmse": evaluator.evaluate(predictions)},
    {"training_release": "als_sensitive_noisy", "rmse": evaluator.evaluate(noisy_predictions)},
])

In [ ]:
ml_release_data = ReleaseData(
    name="spark_als_sensitive_noise_and_month_generalization",
    data=ml_release,
    knowledge_cols=("movie_id", "rating_ml_noisy", "month_ml_generalized"),
    rating_col="rating_ml_noisy",
    description="Spark ALS factor leverage controls rating noise and month generalization.",
)
trials = evaluate_release_linkage_risk(ml_release_data, n_known_values=(1, 2, 3), trials=80, seed=SEED)
summarize_linkage_trials(trials)

## Production notes

- Move the Spark ALS wrapper into `/src` once hyperparameters and diagnostics settle.
- Save factor diagnostics as a report artifact so blue-team release parameters are reproducible.
- Consider `implicit` ALS later if the task becomes implicit feedback rather than explicit rating prediction.